# Module 04 — Distributions, Statistical Significance, Effect Size, Power
Dedicated notebook for Module 04 — reproduces: outlier cleanup, 95% confidence interval, skewness & Shapiro-Wilk, Cohen's d, eta², the sample-size-by-power table.

Human-reviewed English translation of the original Vietnamese notebook (code and results are identical, only text is translated).

SPSS/PSPP equivalents: `Transform > Compute`, `DESCRIPTIVES`, `EXAMINE`, `T-TEST`, `ONEWAY`.

In [ ]:
# Run this cell first (Colab: click ▶). No installation needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED 'Classroom of 240 students' dataset — SAME seed=2026 used
    throughout Modules 03-08, exactly matching lop_hoc_240.csv/.sav in the
    repo (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Male", "Female"], n)
    method = rng.choice(["Traditional", "Project-based"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Project-based")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Project-based") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Created df:", df.shape)

## 0. Clean the `study_hours` outlier BEFORE analysing
One student has `study_hours=48` — almost certainly a typo for "4.8" entered as "48" (see Module 01). Skipping this step would give a skewness of ≈7.4 instead of 0.658.

In [ ]:
mask = df["study_hours"] > 20
print("Rows fixed:", mask.sum(), "| value before:", df.loc[mask, "study_hours"].tolist())
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10
print("value after:", df.loc[mask, "study_hours"].tolist())

## 1. 95% confidence interval for mean Math score
Formula: CI₉₅ = x̄ ± 1.96 × SE, where SE = s/√n.

🎯 **What question does this method answer?** A confidence interval (CI) answers: if the sampling study were repeated many times, what range of values would likely contain the TRUE population parameter. Unlike reporting a single sample mean, the CI also tells you the UNCERTAINTY attached to that number — the smaller the sample, the wider (less certain) the CI.

In [ ]:
m = df["math"].mean(); s = df["math"].std(ddof=1); n = len(df)
se = s / np.sqrt(n)
ci_low, ci_high = m - 1.96*se, m + 1.96*se
print(f"mean={m:.2f}  s={s:.2f}  SE={se:.3f}")
print(f"CI95% = [{ci_low:.2f}; {ci_high:.2f}]")

#### 📤 Actual output
mean=47.91, SE=0.625 → CI95%=[46.68; 49.14]. ✅ Matches the figure in the lesson.

## 2. Skewness & Shapiro-Wilk: `study_hours` (cleaned) vs `pretest`
Rule of thumb: compare skewness to ±2×SE(skewness), SE(skew) ≈ √(6/n).

🎯 **What question does this method answer?** Skewness/Shapiro-Wilk testing answers: is the data close enough to a normal distribution to justify using PARAMETRIC tests (t-test, ANOVA — which assume normality)? This is a check of the METHOD'S ASSUMPTIONS, not a research result in itself — if badly violated, switch to non-parametric tests (Module 07).

In [ ]:
se_skew = np.sqrt(6/n)
for col in ["study_hours", "pretest"]:
    sk = stats.skew(df[col], bias=False)
    w, p = stats.shapiro(df[col])
    verdict = "MEANINGFULLY skewed" if abs(sk) > 2*se_skew else "within the acceptable range"
    print(f"{col}: skew={sk:+.3f} (threshold ±{2*se_skew:.3f}, {verdict}) | Shapiro-Wilk W={w:.3f}, p={p:.4f}")

#### 📤 Actual output
`study_hours`: skew=+0.658, threshold ±0.314 → EXCEEDS the threshold, Shapiro-Wilk p<0.001 (does not support a "normal data" hypothesis). `pretest`: skew=-0.097, within range, Shapiro-Wilk p=0.117 (insufficient evidence against normality). ✅ Matches the lesson.

## 3. Cohen's d — score gain by teaching method

🎯 **What question does this method answer?** Cohen's d answers: how LARGE is the difference between two groups, measured in “number of standard deviations” — it complements the p-value (which only says statistically significant/not) with an answer about the PRACTICAL size, something the p-value alone cannot tell you.

In [ ]:
df["gain"] = df["posttest"] - df["pretest"]
g1 = df.loc[df["method"]=="Project-based", "gain"]
g2 = df.loc[df["method"]=="Traditional", "gain"]
sp = np.sqrt(((len(g1)-1)*g1.var(ddof=1) + (len(g2)-1)*g2.var(ddof=1)) / (len(g1)+len(g2)-2))
d = (g1.mean() - g2.mean()) / sp
t, p = stats.ttest_ind(g1, g2)
print(f"Project-based: n={len(g1)}, M={g1.mean():.2f}, SD={g1.std(ddof=1):.2f}")
print(f"Traditional: n={len(g2)}, M={g2.mean():.2f}, SD={g2.std(ddof=1):.2f}")
print(f"t={t:.3f}, p={p:.2e}")
print(f"Cohen's d = {d:.3f}")

#### 📤 Actual output
d≈0.981 (medium-to-large, close to the "large" threshold of 1.00), t=7.589 (p<0.001). ✅ Matches the lesson.

## 4. Eta² — school → Math score (paired with ANOVA in Module 06)

🎯 **What question does this method answer?** Eta² answers: what percentage of the variance in the dependent variable is “explained” by the grouping variable — similar to R² in regression, this is an EFFECT SIZE specifically for ANOVA, complementing the p-value (which only says significant/not, not how large).

In [ ]:
grand = df["math"].mean()
ssb = sum(len(g)*(g["math"].mean()-grand)**2 for _, g in df.groupby("school"))
sst = ((df["math"]-grand)**2).sum()
eta2 = ssb / sst
print(f"SSB={ssb:.2f}  SST={sst:.2f}  eta²={eta2:.4f}")

#### 📤 Actual output
eta²≈0.047 → a SMALL effect, even though the F-test is statistically significant (p=0.003)! Proof that statistical significance and effect size are two different questions.

## 5. Sample-size-by-power table (Cohen, 1988, statsmodels)
Install `statsmodels` if not already present: already installed on Colab.

🎯 **What question does this method answer?** Power analysis answers: how many observations are needed for a REASONABLE chance of detecting a true effect (if one really exists) — done BEFORE collecting data, not after. If the sample is too small, even a real effect is easily missed (Type II error).

In [ ]:
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()
rows = []
for d_val, label in [(0.2,"Small"), (0.5,"Medium"), (0.8,"Large")]:
    n_req = analysis.solve_power(effect_size=d_val, alpha=0.05, power=0.80, alternative="two-sided")
    rows.append((d_val, label, round(n_req)))
pd.DataFrame(rows, columns=["Effect size d", "Level", "Sample size/group"])

#### 📤 Actual output
d=0.2→≈394/group, d=0.5→≈64/group, d=0.8→≈26/group (α=0.05, power=0.80). ✅ Matches the table in Modules 02 and 04.